In [2]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

# Update data_path to reflect the local file path
data_path = "/Users/bonsitukebeto/Library/CloudStorage/OneDrive-SharedLibraries-NorthwesternUniversity/Arvind Krishna - Data/All Calls by Month"
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
i = 0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows=5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows=5))
    print(i, f.stem, df[i].shape)
    i += 1

# Identify common columns
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)
print(common_cols)

### Read all data files
df_main = pd.DataFrame(columns=list(common_cols))
i = 0
for f in files:
    if f.suffix.lower() == ".csv":
        dfi = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        dfi = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, dfi], ignore_index=True)
    print(i, f.stem, dfi.shape)
    i += 1

### Datetime conversion
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

df_allcallsdata = df_main.copy()

# Step 1: sort preview
df_allcallsdata.sort_values(by=['Correlation ID', 'Start time'], ascending=[True, True])[
    ['Correlation ID', 'Start time', 'Called number', 'Duration']
].head(50)

# Step 2: inbound calls
df_allcallsdata["Duration"] = pd.to_numeric(df_allcallsdata["Duration"], errors="coerce")
df_allcallsdata = df_allcallsdata[df_allcallsdata["Duration"] > 0]
df_allcallsdata = df_allcallsdata.sort_values(by=["Correlation ID", "Start time"])

def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "NA":
        return "Internal"
    else:
        return "Other"

df_allcallsdata["TempCallType"] = df_allcallsdata.apply(classify_call, axis=1)
earliest_calltype = df_allcallsdata.groupby("Correlation ID").first().reset_index()[["Correlation ID", "TempCallType"]]
df_allcallsdata = df_allcallsdata.drop(columns=["TempCallType"])
df_allcallsdata = df_allcallsdata.merge(
    earliest_calltype.rename(columns={"TempCallType": "Inbound/Outbound"}),
    on="Correlation ID", how="left"
)

df_allcallsdata_inbound = df_allcallsdata[df_allcallsdata["Inbound/Outbound"] == "Inbound"].copy()

# Time features
df_allcallsdata_inbound["Start time"] = pd.to_datetime(df_allcallsdata_inbound["Start time"])
df_allcallsdata_inbound["Hour"] = df_allcallsdata_inbound["Start time"].dt.hour
df_allcallsdata_inbound["DayOfWeek"] = df_allcallsdata_inbound["Start time"].dt.weekday + 1
df_allcallsdata_inbound["Month"] = df_allcallsdata_inbound["Start time"].dt.month
df_allcallsdata_inbound["Quarter"] = df_allcallsdata_inbound["Start time"].dt.quarter
df_allcallsdata_inbound["Year"] = df_allcallsdata_inbound["Start time"].dt.year

# Step 3: number descriptions
number_map = {
    "13123478300": "Internal voicemail - not client related",
    "13123411070": "Main number",
    "13124235938": "Community Legal Clinics",
    "13122296300": "Direct Line to Front Desk",
    "1180": "Transfers to English Queue Options (Legal Menu)",
    "13125068646": "Transfers to the English main menu",
    "13124312299": "Farmworker main number / Migrant Legal Assistance Program",
    "13122296079": "Nursing Home Ombudsman",
    "13122296344": "Bankruptcy Helpdesk Voicemail",
    "13122296071": "Criminal Records",
    "13125068647": "Transfers to the Spanish main menu",
    "13123478340": "Veterans Rights Project Voicemail",
    "13122296014": "Markham Eviction Help Desk",
    "13123478309": "HIV Intake Voicemail",
    "13122296072": "Juvenile Expungement Help Desk (JEHD)",
    "13124235904": "Austin Intake Voicemail",
    "13123478347": "A2J Immigration (Lisa Palumbo's direct line)",
    "18882652188": "A2J Immigration (Lisa Palumbo's direct line)",
    "13124235900": "CLASP Voicemail",
    "13123478392": "Education Law Referrals Voicemail",
    "13124235909": "Fair Housing Intake Voicemail",
    "13124312101": "OP Appeals Project",
    "13122296073": "Trafficking Survivors Assistance Project (TSAP)",
    "18004459025": "Migrant Legal Assistance Program",
    "18884018200": "Nursing Home Ombudsman"
}

df_allcallsdata_inbound["Number Description"] = (
    df_allcallsdata_inbound["Called number"].astype(str).map(number_map).fillna("Unknown")
)

# Step 4: call flow / legs 1–4
df_allcallsdata_inbound = df_allcallsdata_inbound.sort_values(["Correlation ID", "Start time"])

call_sequences = (
    df_allcallsdata_inbound.groupby("Correlation ID")["Number Description"]
    .apply(lambda x: [v for v in x if pd.notna(v)])
    .reset_index(name="Call Flow")
)
call_sequences["num_legs"] = call_sequences["Call Flow"].apply(len)
call_sequences_clean = call_sequences[call_sequences["num_legs"] <= 4].copy()

max_legs = call_sequences_clean["Call Flow"].apply(len).max()
for i in range(max_legs):
    call_sequences_clean[f"Number Description_{i+1}"] = call_sequences_clean["Call Flow"].apply(
        lambda x: x[i] if len(x) > i else None
    )

# STEP 5: Merge first-leg time fields and add Date column to call_sequences_merged
first_leg_info = df_allcallsdata_inbound.groupby("Correlation ID").first().reset_index()
first_leg_info = first_leg_info[["Correlation ID", "Hour", "DayOfWeek", "Month", "Quarter", "Year", "Start time"]] 
first_leg_info['Date'] = first_leg_info['Start time'].dt.date  # Add Date column

call_sequences_merged = call_sequences_clean.merge(first_leg_info, on="Correlation ID", how="left")

# STEP 6: Average calls per weekday metrics (corrected to count unique calls)
df_weekdays_unique = call_sequences_merged[call_sequences_merged['DayOfWeek'] <= 5].copy()

# Ensure that we have a proper Date column for weekday calculation
num_weekdays = df_weekdays_unique['Date'].nunique()  # Calculate number of weekdays

# Hourly - now counting unique calls (using 'Correlation ID' to count unique calls)
hourly_totals = df_weekdays_unique.groupby('Hour')['Correlation ID'].nunique().reset_index(name='Total_Calls_This_Hour')
hourly_totals['Avg_Calls_Per_Weekday_Hourly'] = hourly_totals['Total_Calls_This_Hour'] / num_weekdays

# Merge hourly totals back to call_sequences_merged
call_sequences_merged = call_sequences_merged.merge(
    hourly_totals[['Hour', 'Avg_Calls_Per_Weekday_Hourly']], on='Hour', how='left'
)

# Daily - Calculate unique calls per weekday (DayOfWeek)
day_counts = df_weekdays_unique.groupby('DayOfWeek')['Date'].nunique().reset_index(name='Num_Occurrences')
daily_totals = df_weekdays_unique.groupby('DayOfWeek')['Correlation ID'].nunique().reset_index(name='Total_Calls_This_Day')
daily_totals = daily_totals.merge(day_counts, on='DayOfWeek')
daily_totals['Avg_Calls_Per_Weekday_Daily'] = (
    daily_totals['Total_Calls_This_Day'] / daily_totals['Num_Occurrences']
)

# Merge daily totals back to call_sequences_merged
call_sequences_merged = call_sequences_merged.merge(
    daily_totals[['DayOfWeek', 'Avg_Calls_Per_Weekday_Daily']], on='DayOfWeek', how='left'
)

# Monthly (Month, Year) - calculate unique calls per month
monthly_totals = df_weekdays_unique.groupby(['Month', 'Year']).agg(
    Total_Calls_This_Month=('Correlation ID', 'nunique'),
    Weekdays_This_Month=('Date', 'nunique')
).reset_index()
monthly_totals['Avg_Calls_Per_Weekday_Monthly'] = (
    monthly_totals['Total_Calls_This_Month'] / monthly_totals['Weekdays_This_Month']
)

# Merge monthly totals back to call_sequences_merged
call_sequences_merged = call_sequences_merged.merge(
    monthly_totals[['Month', 'Year', 'Avg_Calls_Per_Weekday_Monthly']],
    on=['Month', 'Year'], how='left'
)

# Quarterly (Quarter, Year) - calculate unique calls per quarter
quarterly_totals = df_weekdays_unique.groupby(['Quarter', 'Year']).agg(
    Total_Calls_This_Quarter=('Correlation ID', 'nunique'),
    Weekdays_This_Quarter=('Date', 'nunique')
).reset_index()
quarterly_totals['Avg_Calls_Per_Weekday_Quarterly'] = (
    quarterly_totals['Total_Calls_This_Quarter'] / quarterly_totals['Weekdays_This_Quarter']
)

# Merge quarterly totals back to call_sequences_merged
call_sequences_merged = call_sequences_merged.merge(
    quarterly_totals[['Quarter', 'Year', 'Avg_Calls_Per_Weekday_Quarterly']],
    on=['Quarter', 'Year'], how='left'
)

# Export the final results
inbound_call_workflow = call_sequences_merged
inbound_call_workflow.to_csv("Nov5_AllCallsData_Inbound_Calls.csv", index=False)
print("CSV exported: Nov5_AllCallsData_Inbound_Calls.csv")


0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 66)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)
Columns missing from at least one dataframe: {'Public Calling IP Address', 'Call Recording Trigger', 'Call Recording Result', 'Recall Type', 'Call Recording Platform Name', 'Column1', 'Queue Type', 'Public Called IP Address', 'Redirecting party UUID', 'PSTN vendor name2', 'Answered Elsewhere', 'Original called party UUID', 'External caller ID number', 'Original reason2', 'Auto Attendant Key Pressed', 'Hold Duration', 'User', 'Device owner UUID'}
{'Releasing party', 'Remote SessionID', 'Device Mac', 'PSTN vendor name', 'Local call ID', 'Inbound trunk', 'Redirecting number', 'Report time', 'Org